# Robustness to paraphrased and messy input

Does an agent still succeed when the same request is worded differently? Each of the 13 cases is asked three ways, across all 24 variants:

| Round | Wording | Example (EVAL-01) | Input file |
|---|---|---|---|
| Original | As written in the case set | What items were in my order ORD-2026-00102? | `../clean_scored_merged_dedup.csv` |
| Paraphrase | Same meaning, different sentence | Could you tell me what I bought in order ORD-2026-00102? | `paraphrase_clean_scored_merged_dedup_.csv` |
| Messy input | Typos, slang, lowercase, loose IDs | hey wat items r in my order ord 2026 00102 | `messy_clean_scored_merged_dedup_.csv` |

Only the wording changes. Ground truth, expected ticket status, expected writes and expected tool calls stay the same, so a messy order ID such as `ord 2026 00102` must still be normalised to `ORD-2026-00102` in the tool arguments. All variant wordings are in `robustness_cases_overview.csv`.

| Cases | Case type |
|---|---|
| EVAL-01 to EVAL-08 | Simple lookup |
| EVAL-09, EVAL-10, EVAL-12, EVAL-13 | Multi-step (EVAL-12 and EVAL-13 are multi-turn; every turn is reworded) |
| EVAL-11 | Safety and guardrail |

In [1]:
from pathlib import Path
import io
import pandas as pd

CASE_IDS = [f"EVAL-{i:02d}" for i in range(1, 14)]
KEY = ["case_id", "setup", "ticket_id"]
ROUNDS = ["original", "paraphrase", "messy"]
METRICS = {"task": "task_success", "tool": "tool_call_accuracy", "step": "step_efficiency"}

cwd = Path.cwd()
candidates = [
    cwd,
    cwd / "results/deepeval/robustness_evaluation",
    cwd / "GenAIProject/results/deepeval/robustness_evaluation",
]
DIRECTORY = next(
    (p for p in candidates if (p / "cases_paraphrase.csv").exists()),
    None,
)
if DIRECTORY is None:
    raise FileNotFoundError("Open the notebook from the project or its own directory.")

FILES = {
    "original": DIRECTORY.parent / "clean_scored_merged_dedup.csv",
    "paraphrase": DIRECTORY / "paraphrase_clean_scored_merged_dedup_.csv",
    "messy": DIRECTORY / "messy_clean_scored_merged_dedup_.csv",
}
CATEGORIES = pd.read_csv(DIRECTORY.parent / "case_categories.csv")[["case_id", "category"]]

## 1. Read and match the rounds

Records are matched on **case ID + setup + ticket ID**. Every round must contain the same 312 keys (13 cases × 24 variants).

As in the consistency evaluation, only the `*_success_manual` judgment columns are used:

| Outcome | Passes when |
|---|---|
| Task success | `answer_correctness_success_manual` is True **and** `ticket_status == expected_ticket_status` |
| Tool-call accuracy | `tool_correctness_success_manual` **and** `argument_correctness_success_manual` are both True |
| Step efficiency | `step_efficiency_trace_success_manual` is True |

In [2]:
def read_input(path):
    # The baseline has a .csv name but currently contains an Excel workbook.
    content = path.read_bytes()
    if content[:2] == b"PK":
        return pd.read_excel(io.BytesIO(content))
    return pd.read_csv(path)


def as_boolean(values):
    lookup = {"true": True, "false": False, "1": True, "0": False}
    result = values.astype(str).str.strip().str.lower().map(lookup)
    if result.isna().any():
        raise ValueError(f"Missing or invalid manual judgment: {values.name}")
    return result.astype(bool)


def load_round(name, path):
    frame = read_input(path)
    frame = frame.loc[frame.case_id.isin(CASE_IDS)].copy()
    if frame[KEY].isna().any().any() or frame.duplicated(KEY).any():
        raise ValueError(f"{name}: missing or duplicate case/setup/ticket keys")
    status_match = frame.ticket_status.astype(str) == frame.expected_ticket_status.astype(str)
    return pd.DataFrame({
        **{k: frame[k] for k in KEY},
        f"task_pass_{name}": as_boolean(frame.answer_correctness_success_manual) & status_match,
        f"tool_pass_{name}": as_boolean(frame.tool_correctness_success_manual)
                             & as_boolean(frame.argument_correctness_success_manual),
        f"step_pass_{name}": as_boolean(frame.step_efficiency_trace_success_manual),
    })


rounds = {name: load_round(name, path) for name, path in FILES.items()}
data = rounds["original"]
for name in ROUNDS[1:]:
    data = data.merge(rounds[name], on=KEY, how="outer", indicator=True, validate="one_to_one")
    if (data["_merge"] != "both").any():
        raise ValueError(f"{name}: keys do not match the original round")
    data = data.drop(columns="_merge")

data = data.merge(CATEGORIES, on="case_id", how="left")
print(f"Matched {len(data)} case/setup records across {len(ROUNDS)} rounds.")

Matched 312 case/setup records across 3 rounds.


## 2. Robust pass: does the case pass under every wording?

For one case and setup, let o, p and m be its True/False outcomes for the original, paraphrase and messy rounds.

$$
R_i = o \wedge p \wedge m =
\begin{cases}
1 & \text{if all three rounds pass} \\
0 & \text{otherwise}
\end{cases}
$$

| Original | Paraphrase | Messy | Robust pass? | Reading |
|---|---|---|---|---|
| True | True | True | **Yes** | Wording does not matter |
| True | True | False | No | Broken by messy input |
| True | False | True | No | Broken by rewording |
| False | True | True | No | Failed the original case, so nothing to protect |

Apply the rule separately to task success, tool-call accuracy and step efficiency. Unlike consistency, robustness measures **success under every wording**, not only agreement. A case that fails all three rounds is consistent but not robust.

In [3]:
for metric in METRICS:
    cols = [f"{metric}_pass_{r}" for r in ROUNDS]
    data[f"{metric}_robust_pass"] = data[cols].all(axis=1)
    # Passed the original but lost it under at least one wording.
    data[f"{metric}_broken_by_variant"] = data[f"{metric}_pass_original"] & ~data[f"{metric}_robust_pass"]

case_results = data[KEY + ["category"] + [
    c for m in METRICS for c in
    [f"{m}_pass_original", f"{m}_pass_paraphrase", f"{m}_pass_messy", f"{m}_robust_pass"]
]].sort_values(KEY).reset_index(drop=True)
case_results.head()

,case_id,setup,ticket_id,category,task_pass_original,task_pass_paraphrase,task_pass_messy,task_robust_pass,tool_pass_original,tool_pass_paraphrase,tool_pass_messy,tool_robust_pass,step_pass_original,step_pass_paraphrase,step_pass_messy,step_robust_pass
0,EVAL-01,s1-a,TKT-EVAL-02,simple lookup,True,True,True,True,True,True,True,True,True,True,True,True
1,EVAL-01,s1-b,TKT-EVAL-02,simple lookup,True,True,True,True,True,True,True,True,True,True,True,True
2,EVAL-01,s2-a,TKT-EVAL-02,simple lookup,True,True,True,True,True,True,True,True,True,True,True,True
3,EVAL-01,s2-b,TKT-EVAL-02,simple lookup,False,False,False,False,True,True,True,True,True,True,False,False
4,EVAL-01,s3-a,TKT-EVAL-02,simple lookup,True,True,True,True,True,True,True,True,True,True,True,True


## 3. Combine cases into one summary row per setup

Let **N = 13** be the number of cases for each setup.

$$
\text{Robust pass rate (\%)}=\frac{\text{Cases that pass in all three rounds}}{N}\times100\%
$$

The robust pass rate can never exceed the original pass rate. To separate how good a setup is from how sensitive it is to wording, the summary also reports **retention**: of the cases a setup passed originally, the share that still pass under both rewordings.

$$
\text{Retention (\%)}=\frac{\text{Cases that pass in all three rounds}}{\text{Cases that pass in the original round}}\times100\%
$$

For example, if a setup passes **11 of 13** cases originally and **10 of 13** survive both rewordings, the robust pass rate is **76.92%** and retention is **90.91%**.

| Summary column | What is counted? |
|---|---|
| `<metric>_original_pct` | Cases passing in the original round |
| `<metric>_paraphrase_pct` / `<metric>_messy_pct` | Cases passing in that variant round on its own |
| `<metric>_robust_pct` | Cases passing in all three rounds |
| `<metric>_retention_pct` | Robust cases ÷ originally passing cases |

In [4]:
def summarise(frame, by):
    records = []
    for name, group in frame.groupby(by, sort=True):
        result = {by: name, "cases": len(group)}
        for metric, label in METRICS.items():
            original = group[f"{metric}_pass_original"].sum()
            robust = group[f"{metric}_robust_pass"].sum()
            result[f"{label}_original_pct"] = original / len(group) * 100
            result[f"{label}_paraphrase_pct"] = group[f"{metric}_pass_paraphrase"].mean() * 100
            result[f"{label}_messy_pct"] = group[f"{metric}_pass_messy"].mean() * 100
            result[f"{label}_robust_pct"] = robust / len(group) * 100
            result[f"{label}_retention_pct"] = robust / original * 100 if original else None
        records.append(result)
    return pd.DataFrame(records)


setup_summary = summarise(data, "setup")
assert (setup_summary.cases == len(CASE_IDS)).all(), "every setup needs all 13 cases"
robust_cols = ["setup"] + [f"{label}_robust_pct" for label in METRICS.values()]
print(setup_summary[robust_cols].round(2).to_string(index=False))

setup  task_success_robust_pct  tool_call_accuracy_robust_pct  step_efficiency_robust_pct
 s1-a                    53.85                          84.62                       76.92
 s1-b                    92.31                          84.62                       92.31
 s2-a                    84.62                          76.92                       92.31
 s2-b                    76.92                          84.62                       92.31
 s3-a                    69.23                          69.23                       92.31
 s3-b                    76.92                          61.54                       92.31
 s3-c                    84.62                          84.62                      100.00
 s3-d                    92.31                          76.92                       84.62
 s4-a                    84.62                          69.23                       92.31
 s4-b                    76.92                          76.92                       84.62
 s4-c     

## 4. Overall view: where does robustness drop?

All 312 case/setup records pooled, one row per outcome. The drop columns show how many percentage points each wording costs compared with the original round.

In [5]:
overall = []
for metric, label in METRICS.items():
    row = {"metric": label}
    for r in ROUNDS:
        row[f"{r}_pct"] = data[f"{metric}_pass_{r}"].mean() * 100
    row["robust_pct"] = data[f"{metric}_robust_pass"].mean() * 100
    row["paraphrase_drop_pts"] = row["original_pct"] - row["paraphrase_pct"]
    row["messy_drop_pts"] = row["original_pct"] - row["messy_pct"]
    row["retention_pct"] = data[f"{metric}_robust_pass"].sum() / data[f"{metric}_pass_original"].sum() * 100
    overall.append(row)
overall = pd.DataFrame(overall)
overall.round(2)

,metric,original_pct,paraphrase_pct,messy_pct,robust_pct,paraphrase_drop_pts,messy_drop_pts,retention_pct
0,task_success,86.22,85.26,83.65,81.09,0.96,2.56,94.05
1,tool_call_accuracy,83.33,83.01,78.21,75.64,0.32,5.13,90.77
2,step_efficiency,95.83,95.51,92.63,91.67,0.32,3.21,95.65


## 5. Robustness by case category

Multi-step and multi-turn cases carry more steps where a reworded or misspelled request can be misread, so they are expected to lose more than single lookups.

In [6]:
category_summary = summarise(data, "category")
category_summary[["category", "cases"] + [
    f"{label}_{k}_pct" for label in METRICS.values() for k in ("original", "robust", "retention")
]].round(2)

,category,cases,task_success_original_pct,task_success_robust_pct,task_success_retention_pct,tool_call_accuracy_original_pct,tool_call_accuracy_robust_pct,tool_call_accuracy_retention_pct,step_efficiency_original_pct,step_efficiency_robust_pct,step_efficiency_retention_pct
0,multi-step,96,66.67,59.38,89.06,71.88,58.33,81.16,89.58,86.46,96.51
1,safety & guardrail,24,100.00,95.83,95.83,100.00,100.00,100.00,91.67,79.17,86.36
2,simple lookup,192,94.27,90.10,95.58,86.98,81.25,93.41,99.48,95.83,96.34


## 6. Which case/setup records were broken by rewording?

Records that passed the original round but failed at least one variant round. These are the concrete robustness failures to read in the judge reasons of the variant scored files.

In [7]:
broken = []
for metric, label in METRICS.items():
    mask = data[f"{metric}_broken_by_variant"]
    part = data.loc[mask, KEY + ["category"]].copy()
    part["metric"] = label
    part["failed_paraphrase"] = ~data.loc[mask, f"{metric}_pass_paraphrase"]
    part["failed_messy"] = ~data.loc[mask, f"{metric}_pass_messy"]
    broken.append(part)
broken = pd.concat(broken).sort_values(["metric"] + KEY).reset_index(drop=True)
print(f"{len(broken)} metric-level robustness failures")
broken.groupby(["metric", "case_id"]).size().unstack(fill_value=0)

53 metric-level robustness failures


case_id,EVAL-01,EVAL-02,EVAL-03,EVAL-04,EVAL-05,EVAL-06,EVAL-07,EVAL-08,EVAL-09,EVAL-10,EVAL-11,EVAL-12,EVAL-13
metric,,,,,,,,,,,,,
step_efficiency,1,0,3,1,0,1,0,1,1,1,3,0,1
task_success,1,0,1,1,2,1,1,1,1,3,1,2,1
tool_call_accuracy,0,2,1,3,2,0,0,3,2,5,0,2,4


## 7. Save the results

| Output | Contents |
|---|---|
| `case_robustness_per_case.csv` | One row per case and variant: pass/fail for each round and the robust pass, per metric |
| `final_robustness_summary.csv` | Per-setup original, paraphrase, messy and robust pass rates plus retention |

The input files are unchanged. Rerun all cells after changing files or cases.

The variant rounds are produced with the same runner and judge as the baseline (configured as in `deepeval/README.md`), once per variant:

```sh
python deepeval/run.py --cases results/deepeval/robustness_evaluation/cases_paraphrase.csv \
    --setup s1-a --setup s1-b ... --output results/deepeval/robustness_evaluation/paraphrase_results.jsonl
python deepeval/evaluate.py --input results/deepeval/robustness_evaluation/paraphrase_results.jsonl \
    --output results/deepeval/robustness_evaluation/paraphrase_scored.csv
```

Repeat with `cases_messy.csv`, review the `*_success_manual` columns as for the baseline, and point `FILES` at the scored CSVs.

In [8]:
case_results.to_csv(DIRECTORY / "case_robustness_per_case.csv", index=False)
setup_summary.to_csv(DIRECTORY / "final_robustness_summary.csv", index=False)
print("Saved the two result tables.")

Saved the two result tables.
